In [38]:
import pandas as pd
from pathlib import Path

paths = {
    'profiles': Path('../CSV/19.01.2026/profiles.csv'),
    'predictions': Path('../CSV/01.04.2026/predictions_rows.csv'),
    'fixtures': Path('../CSV/01.04.2026/fixtures_rows.csv'),
    'game_weeks': Path('../CSV/01.04.2026/game_weeks_rows.csv'),
    'season_players': Path('../CSV/25.03.2026/season_players_rows.csv'),
}

def load_and_clean(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    df = pd.read_csv(p, dtype=str)
    # strip whitespace from string columns
    df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
    # coerce obvious datetime-like columns
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ('date','time','created_at','kick','kickoff','updated_at','predictions_close')):
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

# Load dataframes
profiles = load_and_clean(paths['profiles'])
predictions = load_and_clean(paths['predictions'])
fixtures = load_and_clean(paths['fixtures'])
game_weeks = load_and_clean(paths['game_weeks'])
season_players = load_and_clean(paths['season_players'])

# Quick checks
print('profiles', profiles.shape)
print('predictions', predictions.shape)
print('fixtures', fixtures.shape)
print('game_weeks', game_weeks.shape)
print('season_players', season_players.shape)

profiles (48, 4)
predictions (10140, 7)
fixtures (300, 8)
game_weeks (30, 8)
season_players (34, 4)


In [39]:
# Merge per spec:
# profiles.id -> predictions.user_id
# predictions.fixture_id -> fixtures.id
# fixtures.game_week_id -> game_weeks.id

# Merge profiles into predictions
if 'user_id' in predictions.columns and 'id' in profiles.columns:
    pred_profiles = predictions.merge(profiles, left_on='user_id', right_on='id', how='left', suffixes=('','_profile'))
else:
    pred_profiles = predictions.copy()

# Merge predictions -> fixtures
if 'fixture_id' in pred_profiles.columns and 'id' in fixtures.columns:
    pred_profiles_fixtures = pred_profiles.merge(fixtures, left_on='fixture_id', right_on='id', how='left', suffixes=('','_fixture'))
else:
    pred_profiles_fixtures = pred_profiles.copy()

# Merge fixtures -> game_weeks
if 'game_week_id' in pred_profiles_fixtures.columns and 'id' in game_weeks.columns:
    merged = pred_profiles_fixtures.merge(game_weeks, left_on='game_week_id', right_on='id', how='left', suffixes=('','_gameweek'))
else:
    merged = pred_profiles_fixtures.copy()

print('merged shape:', merged.shape)
merged.head(10)

merged shape: (10140, 27)


,id,user_id,fixture_id,home_prediction,away_prediction,created_at,updated_at,id_profile,username,is_host,...,away_score,created_at_fixture,id_gameweek,season_id,week_number,predictions_open,predictions_close,live_start,live_end,created_at_gameweek
0,0002aa63-df75-40cb-b4a8-567cf7a2d868,3d18f1c8-de23-4c6d-b843-8241da5d994e,2609cb03-f16f-449d-8935-1a305eb7590c,0,2,2025-12-29 01:22:50.943946+00:00,2025-12-29 01:22:50.835000+00:00,3d18f1c8-de23-4c6d-b843-8241da5d994e,🍺The Barman,true,...,3,2025-12-29 01:19:48.270031+00:00,b8e9c830-4504-4e15-9aba-2cf1e3b2de9d,27404307-72b4-4518-a32c-ce15384915d9,18,2025-12-28 01:15:00+00,2026-01-02 23:59:00+00:00,2026-01-03 11:15:00+00,2026-01-04 23:16:00+00,2025-12-29 01:19:48.069015+00:00
1,0002f52f-96ae-40dd-85d1-efc54a517ec6,eff7bddc-a865-47c4-95fb-fe4116d8c288,fe89715d-3e68-40c6-8c23-856529a769e4,1,1,2026-03-10 21:35:39.213855+00:00,2026-03-10 21:35:39.154000+00:00,eff7bddc-a865-47c4-95fb-fe4116d8c288,Sid Elliott,false,...,0,2026-03-10 03:28:31.232061+00:00,db3a5288-ba3c-4edf-96b7-6d7bc46e834c,27404307-72b4-4518-a32c-ce15384915d9,28,2026-03-09 03:21:00+00,2026-03-13 23:59:00+00:00,2026-03-14 07:30:00+00,2026-03-16 23:30:00+00,2026-03-10 03:28:30.960142+00:00
2,0004f2ab-78fa-419d-894e-ef835a1f2762,ceb258ce-454a-4b5f-9321-73fd28390495,7c6fcd56-7715-42e1-8e67-c426c112fef8,1,2,2025-09-30 10:23:38.999143+00:00,2025-09-30 19:44:34.051000+00:00,ceb258ce-454a-4b5f-9321-73fd28390495,Stephen O,false,...,2,2025-09-30 00:40:14.076755+00:00,d77ab4fc-0b5e-46dc-8403-2c02eb802dd6,27404307-72b4-4518-a32c-ce15384915d9,7,2025-09-29 01:35:00+00,2025-10-02 22:59:00+00:00,2025-10-03 10:30:00+00,2025-10-05 20:00:00+00,2025-09-30 00:40:13.853434+00:00
3,000a8955-a583-417f-8bd1-39fcbf90be39,857f0ddb-bbf7-490f-89e7-c393913d1937,920d8483-0bc6-447b-b623-cfd88610f404,1,1,2026-01-02 10:46:48.150588+00:00,2026-01-02 10:46:48.073000+00:00,857f0ddb-bbf7-490f-89e7-c393913d1937,Jim Shirley,false,...,4,2025-12-29 01:19:48.270031+00:00,b8e9c830-4504-4e15-9aba-2cf1e3b2de9d,27404307-72b4-4518-a32c-ce15384915d9,18,2025-12-28 01:15:00+00,2026-01-02 23:59:00+00:00,2026-01-03 11:15:00+00,2026-01-04 23:16:00+00,2025-12-29 01:19:48.069015+00:00
4,000b768b-2b9d-4eaa-b267-30e63acd74b8,584d969b-f2c5-4e89-a226-9d5d47bebe23,5027c125-11c7-47db-bff4-7feed614ca7a,1,3,2025-08-19 10:20:56.427118+00:00,2025-08-19 10:20:56.256000+00:00,584d969b-f2c5-4e89-a226-9d5d47bebe23,Steve arnold,false,...,5,2025-08-18 23:36:49.012624+00:00,28c25bca-56fb-487b-b335-f02e00863c6b,27404307-72b4-4518-a32c-ce15384915d9,2,2025-08-18 01:00:00+00,2025-08-21 22:59:00+00:00,2025-08-22 01:00:00+00,2025-08-25 23:30:00+00,2025-08-18 23:36:48.845230+00:00
5,001442d6-0710-48b0-b4f9-b1858e1d4e6b,e2ffaf02-f17f-4be3-b3ee-651d39315af1,ac253518-11bd-4d01-b548-62279fff4ca1,0,2,2025-10-27 08:55:01.221113+00:00,2025-10-30 17:35:42.603000+00:00,e2ffaf02-f17f-4be3-b3ee-651d39315af1,Mjd-⚒️⚒️⚒️,false,...,2,2025-10-27 02:20:06.870033+00:00,28da2862-f00a-44dd-8ab5-a48849d53245,27404307-72b4-4518-a32c-ce15384915d9,10,2025-10-26 04:12:00+00,2025-10-30 23:59:00+00:00,2025-10-31 02:12:00+00,2025-11-03 23:15:00+00,2025-10-27 02:20:06.707350+00:00
6,00146d7d-b426-4de8-86dc-8240c399f910,e04c559e-ca37-400e-b70d-64cc03eb3361,5128edca-8713-4fff-865e-73b96cbddb30,2,1,2025-12-30 19:07:10.252508+00:00,2025-12-30 19:07:10.137000+00:00,e04c559e-ca37-400e-b70d-64cc03eb3361,Graham Dongworth,false,...,0,2025-12-29 01:19:48.270031+00:00,b8e9c830-4504-4e15-9aba-2cf1e3b2de9d,27404307-72b4-4518-a32c-ce15384915d9,18,2025-12-28 01:15:00+00,2026-01-02 23:59:00+00:00,2026-01-03 11:15:00+00,2026-01-04 23:16:00+00,2025-12-29 01:19:48.069015+00:00
7,0016b45b-f084-4486-825c-938ea90d0ffd,182f8371-048a-4f6e-b95d-149f368c3c44,03f17591-9c55-4c93-8131-3ff2a3385c48,3,0,2025-08-07 14:30:45.852443+00:00,2025-08-07 14:30:44.554000+00:00,182f8371-048a-4f6e-b95d-149f368c3c44,Si B,false,...,0,2025-08-07 14:07:51.860107+00:00,e1083685-268d-4376-bbc8-14fe74194d46,27404307-72b4-4518-a32c-ce15384915d9,1,2025-08-06 15:03:00+00,2025-08-1

In [40]:
# Diagnostic merge with explicit suffixes and column listing
# Use the merged dataframe from Cell 2 when available to avoid re-running merges.
if 'merged' in globals():
    merged_diag = merged.copy()
else:
    # Fallback: perform defensive merges with explicit suffixes for inspection
    pred_profiles = predictions.merge(profiles, left_on='user_id', right_on='id', how='left', suffixes=('','_profile')) if 'user_id' in predictions.columns else predictions.copy()
    pred_profiles_fixtures = pred_profiles.merge(fixtures, left_on='fixture_id', right_on='id', how='left', suffixes=('','_fixture')) if 'fixture_id' in pred_profiles.columns else pred_profiles.copy()
    merged_diag = pred_profiles_fixtures.merge(game_weeks, left_on='game_week_id', right_on='id', how='left', suffixes=('','_gameweek')) if 'game_week_id' in pred_profiles_fixtures.columns else pred_profiles_fixtures.copy()
print('merged_diag shape:', getattr(merged_diag, 'shape', None))
print('merged_diag columns:', list(merged_diag.columns))
merged_diag.head(5)

merged_diag shape: (10140, 27)
merged_diag columns: ['id', 'user_id', 'fixture_id', 'home_prediction', 'away_prediction', 'created_at', 'updated_at', 'id_profile', 'username', 'is_host', 'club', 'id_fixture', 'game_week_id', 'fixture_number', 'home_team', 'away_team', 'home_score', 'away_score', 'created_at_fixture', 'id_gameweek', 'season_id', 'week_number', 'predictions_open', 'predictions_close', 'live_start', 'live_end', 'created_at_gameweek']


,id,user_id,fixture_id,home_prediction,away_prediction,created_at,updated_at,id_profile,username,is_host,...,away_score,created_at_fixture,id_gameweek,season_id,week_number,predictions_open,predictions_close,live_start,live_end,created_at_gameweek
0,0002aa63-df75-40cb-b4a8-567cf7a2d868,3d18f1c8-de23-4c6d-b843-8241da5d994e,2609cb03-f16f-449d-8935-1a305eb7590c,0,2,2025-12-29 01:22:50.943946+00:00,2025-12-29 01:22:50.835000+00:00,3d18f1c8-de23-4c6d-b843-8241da5d994e,🍺The Barman,true,...,3,2025-12-29 01:19:48.270031+00:00,b8e9c830-4504-4e15-9aba-2cf1e3b2de9d,27404307-72b4-4518-a32c-ce15384915d9,18,2025-12-28 01:15:00+00,2026-01-02 23:59:00+00:00,2026-01-03 11:15:00+00,2026-01-04 23:16:00+00,2025-12-29 01:19:48.069015+00:00
1,0002f52f-96ae-40dd-85d1-efc54a517ec6,eff7bddc-a865-47c4-95fb-fe4116d8c288,fe89715d-3e68-40c6-8c23-856529a769e4,1,1,2026-03-10 21:35:39.213855+00:00,2026-03-10 21:35:39.154000+00:00,eff7bddc-a865-47c4-95fb-fe4116d8c288,Sid Elliott,false,...,0,2026-03-10 03:28:31.232061+00:00,db3a5288-ba3c-4edf-96b7-6d7bc46e834c,27404307-72b4-4518-a32c-ce15384915d9,28,2026-03-09 03:21:00+00,2026-03-13 23:59:00+00:00,2026-03-14 07:30:00+00,2026-03-16 23:30:00+00,2026-03-10 03:28:30.960142+00:00
2,0004f2ab-78fa-419d-894e-ef835a1f2762,ceb258ce-454a-4b5f-9321-73fd28390495,7c6fcd56-7715-42e1-8e67-c426c112fef8,1,2,2025-09-30 10:23:38.999143+00:00,2025-09-30 19:44:34.051000+00:00,ceb258ce-454a-4b5f-9321-73fd28390495,Stephen O,false,...,2,2025-09-30 00:40:14.076755+00:00,d77ab4fc-0b5e-46dc-8403-2c02eb802dd6,27404307-72b4-4518-a32c-ce15384915d9,7,2025-09-29 01:35:00+00,2025-10-02 22:59:00+00:00,2025-10-03 10:30:00+00,2025-10-05 20:00:00+00,2025-09-30 00:40:13.853434+00:00
3,000a8955-a583-417f-8bd1-39fcbf90be39,857f0ddb-bbf7-490f-89e7-c393913d1937,920d8483-0bc6-447b-b623-cfd88610f404,1,1,2026-01-02 10:46:48.150588+00:00,2026-01-02 10:46:48.073000+00:00,857f0ddb-bbf7-490f-89e7-c393913d1937,Jim Shirley,false,...,4,2025-12-29 01:19:48.270031+00:00,b8e9c830-4504-4e15-9aba-2cf1e3b2de9d,27404307-72b4-4518-a32c-ce15384915d9,18,2025-12-28 01:15:00+00,2026-01-02 23:59:00+00:00,2026-01-03 11:15:00+00,2026-01-04 23:16:00+00,2025-12-29 01:19:48.069015+00:00
4,000b768b-2b9d-4eaa-b267-30e63acd74b8,584d969b-f2c5-4e89-a226-9d5d47bebe23,5027c125-11c7-47db-bff4-7feed614ca7a,1,3,2025-08-19 10:20:56.427118+00:00,2025-08-19 10:20:56.256000+00:00,584d969b-f2c5-4e89-a226-9d5d47bebe23,Steve arnold,false,...,5,2025-08-18 23:36:49.012624+00:00,28c25bca-56fb-487b-b335-f02e00863c6b,27404307-72b4-4518-a32c-ce15384915d9,2,2025-08-18 01:00:00+00,2025-08-21 22:59:00+00:00,2025-08-22 01:00:00+00,2025-08-25 23:30:00+00,2025-08-18 23:36:48.845230+00:00


In [41]:
# Tidy game-week summary + missing-players table
# Set a target game_week id (or leave placeholder to auto-select the first available id)
targeted_game_week = 'd6d48c4a-c6a6-4149-a03a-6a45d592346a'
if 'XXXX' in str(targeted_game_week) or str(targeted_game_week).strip() == '':
    if not game_weeks.empty and 'id' in game_weeks.columns:
        targeted_game_week = game_weeks['id'].astype(str).iat[0]
    else:
        raise RuntimeError('game_weeks is empty or missing id; set targeted_game_week to a valid id')

# Locate the game-week row
gw_row = game_weeks[game_weeks['id'].astype(str) == str(targeted_game_week)]
if gw_row.empty:
    raise KeyError(f'Game week id {targeted_game_week} not found in game_weeks')
gw = gw_row.iloc[0]
week_number = gw.get('week_number') if 'week_number' in gw_row.columns else None
close_time = gw.get('predictions_close')
close_ts = pd.to_datetime(close_time, errors='coerce') if pd.notna(close_time) else None

# Players in season
season_id = gw.get('season_id')
players_in_season = season_players[season_players['season_id'].astype(str) == str(season_id)]['player_id'].astype(str).unique()
players_count = len(players_in_season)

# Fixtures and predictions counts
fixtures_in_gw = fixtures[fixtures['game_week_id'].astype(str) == str(targeted_game_week)]
fixture_ids = fixtures_in_gw['id'].astype(str).unique()
fixture_count = len(fixture_ids)
gw_preds = predictions[predictions['fixture_id'].astype(str).isin(fixture_ids)].copy()
predictions_count = gw_preds.shape[0]

# Timely predictions (updated_at missing = treated as timely)
gw_preds['updated_at'] = pd.to_datetime(gw_preds['updated_at'], errors='coerce')
if pd.isna(close_ts):
    timely_users = set(gw_preds['user_id'].dropna().astype(str).unique())
else:
    timely_mask = gw_preds['updated_at'].isna() | (gw_preds['updated_at'] <= close_ts)
    timely_users = set(gw_preds.loc[timely_mask, 'user_id'].dropna().astype(str).unique())
timely_count = len(timely_users)

# Summary table (tidy)
summary = pd.DataFrame([{
    'game_week_id': targeted_game_week,
    'week_number': week_number,
    'predictions_close': close_ts,
    'players_in_season': players_count,
    'fixture_count': fixture_count,
    'predictions_count': predictions_count,
    'timely_users_count': timely_count
}])
from IPython.display import display
print('Game-week summary:')
display(summary)

# Missing players table (keep original tidy output)
missing_ids = sorted([m for m in map(str, players_in_season) if m not in timely_users])
missing_profiles = profiles[profiles['id'].astype(str).isin(missing_ids)][['id', 'username']].copy()
print(f'Players missing timely predictions for game_week {targeted_game_week}: {len(missing_profiles)}')
display(missing_profiles.reset_index(drop=True))

Game-week summary:


,game_week_id,week_number,predictions_close,players_in_season,fixture_count,predictions_count,timely_users_count
0,d6d48c4a-c6a6-4149-a03a-6a45d592346a,30,2026-04-01 22:59:00+00:00,34,10,270,27


Players missing timely predictions for game_week d6d48c4a-c6a6-4149-a03a-6a45d592346a: 7


,id,username
0,25472e3e-4928-440c-84a8-ed85a6e4cf34,Matt Lavery
1,298651b6-b5d4-4875-a636-799edae79a84,KAV
2,6a57188f-50ad-4d66-88e5-64236a293c32,Lenny Wright
3,abe5e91c-2516-47f8-98a7-2b1e70ab13b1,Chris Torode
4,c2011527-da58-4568-97f9-ec5c223ead8e,MattyCdogg
5,eb3628cb-9a74-4281-bc9f-ebff66ae268e,MG
6,f9aa08c8-da66-4064-a39a-3225dadaa1d0,Mel
